<div class="alert alert-info">
    <b>1.1 Datenqualität: Ionenbilanzfehler (Rohdaten - Main-Ions-Six)</b><br>
    Dieses Notebook analysiert den Ionenbilanzfehler (IBE) basierend auf den 6 klassischen Hauptionen.<br>
    -> Berücksichtigt NUR: Na, Ca, Mg (Kationen) & Cl, HCO3, SO4 (Anionen)<br>
    -> KEINE Berücksichtigung von pH, K oder NO3<br>
    -> Erstellung eines PDF-Berichts mit Histogrammen und Zusammenfassung
</div>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import os
import glob
import re
import textwrap

# ------------------------- Konfiguration der Pfade -------------------------
INPUT_DIR = "../../../../../../1_Acquisition/1.1_Data-Acquisition-Wrapper/Gesammelte_Datenbanken"
OUTPUT_PDF = "Raw_Data_IBE_Report.pdf"

<div class="alert alert-info">
    <b>Master-Lexikon der Ionen (Reduziert auf 6 Hauptionen)</b><br>
    -> Definition der 6 Kern-Spezies (Na, Ca, Mg, Cl, HCO3, SO4)<br>
    -> Ausschluss von K, NO3 und H+(pH)<br>
    -> Regex-Muster für diverse Einheiten (mg/l, meq/l, etc.)
</div>

In [2]:
# ------------------------- Master Chemisches Wörterbuch (Nur 6 Hauptionen) -------------------------
MASTER_ION_DICT = {
    # ------------------------- Kationen (Haupt) -------------------------
    'Na': {'mass': 22.98977, 'valence': 1, 'type': 'cation', 'names': ['Na', 'Sodium', 'Natrium']},
    'Ca': {'mass': 40.078, 'valence': 2, 'type': 'cation', 'names': ['Ca', 'Calcium']},
    'Mg': {'mass': 24.305, 'valence': 2, 'type': 'cation', 'names': ['Mg', 'Magnesium']},
    
    # ------------------------- Anionen (Haupt) -------------------------
    'Cl': {'mass': 35.45, 'valence': 1, 'type': 'anion', 'names': ['Cl', 'Chloride', 'Chlorid']},
    'HCO3': {'mass': 61.0168, 'valence': 1, 'type': 'anion', 'names': ['HCO3', 'Bicarbonate', 'Alkalinity']},
    'SO4': {'mass': 96.06, 'valence': 2, 'type': 'anion', 'names': ['SO4', 'Sulfate', 'Sulphate']},
}

# ------------------------- Regex für Einheiten -------------------------
UNIT_PATTERNS = {
    'meq/l': r'meq/l|meq|meq-l',
    'mg/l': r'mg/l|ppm|mg|mg-l|mg l-1',
    'mmol/l': r'mmol/l|mm|mmol',
    'umol/l': r'umol/l|\u00b5mol/l|umol',
    'ug/l': r'ug/l|ppb|ug'
}

<div class="alert alert-info">
    <b>Hilfsfunktionen</b><br>
    -> Parsen der Spaltenüberschriften (Suche nach Ion + Einheit)<br>
    -> Umrechnung verschiedener Einheiten in meq/L
</div>

In [3]:
def parse_column_header(col_name):

    col_lower = str(col_name).lower().strip()
    
    # ------------------------- 1. Einheit erkennen -------------------------
    found_unit = None
    for unit_key, pattern in UNIT_PATTERNS.items():
        if re.search(pattern, col_lower):
            found_unit = unit_key
            break
    
    # ------------------------- 2. Ion erkennen -------------------------
    found_ion = None
    best_match_len = 0
    
    for ion_key, info in MASTER_ION_DICT.items():
        for name in info['names']:
            # ------------------------- Schreibweise und Trennung -------------------------
            if len(name) < 2:
                 pattern = rf"(^|[^a-z]){name.lower()}([^a-z]|$)"
            else:
                 pattern = rf"(^|[^a-z]){re.escape(name.lower())}([^a-z]|$)"
            
            if re.search(pattern, col_lower):
                if len(name) > best_match_len:
                    found_ion = ion_key
                    best_match_len = len(name)
    
    if found_ion:
        return found_ion, found_unit
    return None


def get_conversion_factor(unit, molecule_mass, valence):
    # ------------------------- Berechnet den Faktor zur Umrechnung in meq/L -------------------------
    if unit == 'meq/l': return 1.0
    elif unit == 'mg/l': return valence / molecule_mass
    elif unit == 'mmol/l': return valence
    elif unit == 'umol/l': return valence / 1000.0
    elif unit == 'ug/l': return (valence / molecule_mass) / 1000.0
    return 0

<div class="alert alert-info">
    <b>Dynamische IBE-Berechnung (Stripped)</b><br>
    -> Sucht NUR die 6 definierten Hauptionen<br>
    -> KEINE pH/H+ Berechnung
</div>

In [4]:
def calculate_ibe_dynamic(df):
    cations_sum = 0
    anions_sum = 0
    used_ions = {}
    
    # ------------------------- Ionen parsen -------------------------
    for col in df.columns:
        res = parse_column_header(col)
        if res:
            ion, unit = res
            if not unit: unit = 'mg/l'
            
            # ------------------------- Priorisierung: meq/l bevorzugen -------------------------
            if ion in used_ions:
                if unit == 'meq/l' and used_ions[ion]['unit'] != 'meq/l':
                    used_ions[ion] = {'col': col, 'unit': unit}
            else:
                used_ions[ion] = {'col': col, 'unit': unit}
                
    # ------------------------- Werte summieren -------------------------
    display_list = []
    
    for ion, data in used_ions.items():
        col = data['col']
        unit = data['unit']
        info = MASTER_ION_DICT[ion]
        
        factor = get_conversion_factor(unit, info['mass'], info['valence'])
        
        vals = pd.to_numeric(df[col], errors='coerce').fillna(0)
        meq_vals = vals * factor
        
        if info['type'] == 'cation':
            cations_sum += meq_vals
        else:
            anions_sum += meq_vals
            
        display_list.append(f"{ion}")

    # ------------------------- IBE Formel -------------------------
    total = cations_sum + anions_sum
    valid = total > 0.1
    
    ibe = pd.Series(np.nan, index=df.index)
    ibe[valid] = ((cations_sum[valid] - anions_sum[valid]) / total[valid]) * 100
    
    return ibe.dropna(), display_list

<div class="alert alert-info">
    <b>Hauptverarbeitung & Bericht</b><br>
    -> Iteration über alle gefundenen Datenbanken<br>
    -> Erstellung der Histogramme pro Datei<br>
    -> Generierung der finalen Zusammenfassungstabelle (Unusable Data)
</div>

In [5]:
files = glob.glob(os.path.join(INPUT_DIR, "*"))
excel_files = [f for f in files if f.endswith(('.xlsx', '.xls', '.csv'))]

print(f"Scanne {len(excel_files)} Dateien...")

# ------------------------- Globale Zähler -------------------------
global_total_records = 0
global_good_records = 0
db_stats = [] 

# ------------------------- PDF Erstellung -------------------------
with PdfPages(OUTPUT_PDF) as pdf:
    for filepath in excel_files:
        filename = os.path.basename(filepath)
        try:
            # ------------------------- Datei einlesen -------------------------
            if filepath.endswith('.csv'):
                try: df = pd.read_csv(filepath, encoding='utf-8', on_bad_lines='skip')
                except: df = pd.read_csv(filepath, sep=None, engine='python', encoding='latin1')
            else:
                df = pd.read_excel(filepath)
            
            # ------------------------- IBE Berechnen -------------------------
            ibe, found = calculate_ibe_dynamic(df)
            
            # ------------------------- Statistiken -------------------------
            n_total = len(ibe)
            n_good = (ibe.abs() <= 5).sum()
            
            global_total_records += n_total
            global_good_records += n_good
            
            db_stats.append({
                'name': filename[:40], 
                'total': n_total,
                'good': n_good
            })
            
            # ------------------------- Plotting -------------------------
            plt.figure(figsize=(10, 7))
            if len(ibe) > 0:
                plt.hist(ibe, bins=30, range=(-50, 50), edgecolor='black', alpha=0.7)
                plt.title(f"IBE (6 Main Ions): {filename}\n(N={n_total})")
                plt.xlabel("Ionenbilanzfehler (%)")
                plt.ylabel("Anzahl")
                plt.grid(True, alpha=0.3)
                
                # ------------------------- Ionenliste formatieren -------------------------
                found.sort()
                ions_str = ", ".join(found)
                wrapped_ions = textwrap.fill(ions_str, width=65)
                
                # ------------------------- Statistik-Text -------------------------
                mean_val = ibe.mean()
                median_val = ibe.median()
                pct_5 = (n_good / n_total * 100) if n_total > 0 else 0
                
                stats_str = (
                    f"Mean: {mean_val:.2f}%\n"
                    f"Median: {median_val:.2f}%\n"
                )
                
                stats_5pct = f"+/- 5%: {pct_5:.1f}% ({n_good}/{n_total})"
                
                ions_block = f"\n\nErkannt ({len(found)}):\n{wrapped_ions}"
                
                # ------------------------- Text platzieren -------------------------
                plt.text(1.02, 1.0, stats_str, transform=plt.gca().transAxes, 
                         fontsize=10, verticalalignment='top')
                         
                # ------------------------- Highlight 5% Marke -------------------------
                plt.text(1.02, 0.90, stats_5pct, transform=plt.gca().transAxes, 
                         fontsize=11, fontweight='bold', color='red', verticalalignment='top')
                
                plt.text(1.02, 0.85, ions_block, transform=plt.gca().transAxes, 
                         fontsize=8, verticalalignment='top')
                
                plt.tight_layout()
            else:
                plt.text(0.5, 0.5, "Keine Berechnung möglich", ha='center', va='center')
                plt.title(f"{filename} (Keine Daten)")
            
            pdf.savefig()
            plt.close()
            print(f"Verarbeitet: {filename} ({len(found)} Ionen)")
            
        except Exception as e:
            print(f"Fehler {filename}: {e}")
            db_stats.append({'name': filename[:30], 'total': 0, 'good': 0})
    
    # ------------------------- Globale Zusammenfassungsseite -------------------------
    items_per_page = 35 
    total_dbs = len(db_stats)
    
    current_idx = 0
    page_num = 1
    
    print(f"Generiere Zusammenfassung für {total_dbs} Datenbanken...")
    
    while current_idx < total_dbs:
        plt.figure(figsize=(11.69, 8.27)) 
        
        # ------------------------- Fixierte Skalierung um Autoscaling zu verhindern -------------------------
        plt.xlim(0, 1)
        plt.ylim(0, 1)
        plt.axis('off') 
        
        end_idx = min(current_idx + items_per_page, total_dbs)
        
        # ------------------------- Header -------------------------
        plt.text(0.5, 0.95, f"Global Data Quality Summary (Page {page_num})", 
                 fontsize=16, fontweight='bold', ha='center', va='top')
        
        y_pos = 0.90
        
        # ------------------------- Gesamtwert (Seite 1) -------------------------
        if page_num == 1:
            if global_total_records > 0:
                global_pct = (global_good_records / global_total_records) * 100
            else:
                 global_pct = 0
            
            g_text = (f"OVERALL: {global_good_records}/{global_total_records} records good ({global_pct:.1f}%)")
            plt.text(0.5, y_pos, g_text, fontsize=12, fontweight='bold', ha='center', color='blue')
            y_pos -= 0.05
        
        # ------------------------- Spaltenüberschriften -------------------------
        header_str = f"{'Database':<55} | {'Total':<10} | {'+/- 5% (Good)':<15} | {'Unusable':<10}"
        plt.text(0.05, y_pos, header_str, fontsize=10, fontweight='bold', fontfamily='monospace')
        
        plt.plot([0.05, 0.95], [y_pos-0.015, y_pos-0.015], color='black', linewidth=1)
        
        y_pos -= 0.04
        
        # ------------------------- Einträge -------------------------
        for i in range(current_idx, end_idx):
            stat = db_stats[i]
            rem = stat['total'] - stat['good']
            name_disp = (stat['name'][:50] + '..') if len(stat['name']) > 50 else stat['name']
            
            line = f"{name_disp:<55} | {stat['total']:<10} | {stat['good']:<15} | {rem:<10}"
            
            plt.text(0.05, y_pos, line, fontsize=9, fontfamily='monospace')
            y_pos -= 0.022
            
        pdf.savefig()
        plt.close()
        
        current_idx = end_idx
        page_num += 1

print(f"Bericht erstellt: {OUTPUT_PDF}")

Scanne 0 Dateien...
Generiere Zusammenfassung für 0 Datenbanken...
Bericht erstellt: Raw_Data_IBE_Report.pdf


C:\Users\lucca\AppData\Local\Temp\ipykernel_35276\2219958954.py:12: MatplotlibDeprecationWarning: Keeping empty pdf files is deprecated since 3.8 and support will be removed in 3.10.
  with PdfPages(OUTPUT_PDF) as pdf:
